In [ ]:
import os

In [ ]:
_ = sql("select 'Initialize Spark Context'").collect()

In [ ]:
jvm = spark._jvm
gdb = jvm.com.esri.gdb.FileGDB

In [ ]:
jvm.com.esri.spark.Functions.registerFunctions()

In [ ]:
gdb_path = "path/to/GPS.gdb"
# for tab in gdb.listTables(gdb_path):
#     print(tab)

In [ ]:
spark.read \
    .format('com.esri.gdb') \
    .options(path=gdb_path, name='Operation_Record') \
    .load() \
    .createOrReplaceTempView('GPS')

In [ ]:
# sql("describe GPS").show(truncate=True)

In [ ]:
cell_1 = 4.0
cell_2 = cell_1 / 2.0

sql(f"""
    select cast(floor(lon2x(SHAPE.x)/{cell_1}) as int) as x,cast(floor(lat2y(SHAPE.y)/{cell_1}) as int) as y
    from GPS
    where speed > 5
    """) \
    .createOrReplaceTempView("XY")

In [ ]:
sql(f"""
    select x*{cell_1}+{cell_2} as x,y*{cell_1}+{cell_2} as y,count(1) as pop
    from XY
    group by x,y
    having pop > 10
    """)\
    .createOrReplaceTempView("RC")

In [ ]:
%%time

csv = os.path.expanduser(os.path.join("~", "OpRec"))
sql("""
    select pop,concat('POINT(',x,' ',y,')') from RC
""")\
    .write \
    .mode("overwrite") \
    .option("delimiter", "\t") \
    .option("dateFormat", "yyyy-MM-dd HH:mm:ss") \
    .csv(csv)